<a href="https://colab.research.google.com/github/robertbarcik/genai-in-python-tutorial/blob/main/4_fine-tuning/4_fine-tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning: What It Changes, What It Costs, Why You Will Rarely Need It

Your helpdesk bot answers correctly but never quite in the company's voice: too long, too formal, wrong sign-off. A developer message gets you 90% of the way. **Fine-tuning** is the tool for the last 10%: you show the model a few hundred examples of the answers you want, and its weights shift so that this style becomes its default.

This notebook is short on purpose. It shows what actually changes inside the model (with 24 numbers instead of 30 billion), why fine-tuning is for *behaviour* and not *knowledge*, what it costs on three kinds of hardware, and why in 2026 you will reach for it far less often than the hype suggests. No API key needed; everything here runs on your CPU in seconds.

## Setup

In [1]:
%pip install -q numpy   # Colab installs here; locally, `pip install -r requirements.txt` already covers it

import numpy as np
np.set_printoptions(precision=2, suppress=True)
print("Ready. No API key needed in this notebook.")

Note: you may need to restart the kernel to use updated packages.
Ready. No API key needed in this notebook.


## What actually changes inside: LoRA with 24 numbers

A language model is, at bottom, a pile of big matrices. Full fine-tuning would update every number in every matrix: for a 30-billion-parameter model that is 30 billion numbers to store, move and back up.

**LoRA (low-rank adaptation)** does something cheaper. It leaves the original matrix `W` frozen and adds a small correction next to it: two thin matrices `A` and `B` whose product has the same shape as `W`. Training only touches `A` and `B`. Here it is with a 4×4 "model".

In [2]:
rng = np.random.default_rng(0)

W = rng.normal(size=(4, 4))          # the pre-trained weights: FROZEN
A = np.zeros((4, 1))                 # the adapter, rank 1: TRAINED (starts at zero, so at first nothing changes)
B = rng.normal(size=(1, 4)) * 0.1

def model(x, A, B):
    return x @ (W + A @ B)           # the original matrix plus a tiny, learned correction

print("W has", W.size, "numbers (frozen). The adapter A and B have", A.size + B.size, "numbers (trained).")
print("W + A@B has the same shape as W:", (W + A @ B).shape)

W has 16 numbers (frozen). The adapter A and B have 8 numbers (trained).
W + A@B has the same shape as W: (4, 4)


Now a tiny training loop. We want the model to give a different answer for one particular input (our "house style"). Watch two things: the error goes down, and `W` never changes.

In [3]:
x = np.array([[1.0, 0.5, -0.5, 2.0]])          # one training input
target = np.array([[1.0, -1.0, 0.5, 0.0]])       # the output we want it to produce
W_before = W.copy()

for step in range(1, 51):
    prediction = model(x, A, B)
    error = prediction - target
    # gradient descent on A and B only (W is not in this loop at all)
    grad_A = x.T @ (error @ B.T)
    grad_B = (x @ A).T @ error
    A -= 0.05 * grad_A
    B -= 0.05 * grad_B
    if step in (1, 10, 25, 50):
        print(f"step {step:>3}: error = {np.abs(error).mean():.3f}")

print("\nW unchanged:", np.array_equal(W, W_before))
print("prediction now:", model(x, A, B).round(2), " target:", target)

step   1: error = 2.245
step  10: error = 1.596
step  25: error = 0.000
step  50: error = 0.000

W unchanged: True
prediction now: [[ 1.  -1.   0.5  0. ]]  target: [[ 1.  -1.   0.5  0. ]]


### 🔍 What just happened?

The model learned the new behaviour, and the only numbers that moved were the 8 in `A` and `B`. Shipping that fine-tune means shipping 8 numbers, not 16. In real models the matrices are thousands wide and the adapter rank is 8 to 64, which is where these headline numbers come from:

- The LoRA paper (Hu et al., 2021) reports **10,000× fewer trainable parameters** and **3× less GPU memory** than full fine-tuning on GPT-3 175B, with the same quality.
- **QLoRA** (Dettmers et al., 2023) additionally stores the frozen `W` in 4-bit and fine-tunes a **65B model on a single 48 GB GPU**.
- A LoRA adapter for a 30B model is tens of megabytes next to a 60 GB base model, and you can keep several and swap them per customer.

## Behaviour, not knowledge

What did our 8 numbers learn? A *mapping*: this kind of input, that kind of output. That is what fine-tuning is good at: tone, format, a fixed classification scheme, following a house style without being told every time.

What it is bad at: facts. Your price list, this week's outage, the customer's order history. Those change constantly, and a fine-tuned model that memorised last month's prices is confidently wrong this month. Facts go into the **context window at inference time**: a developer message, retrieved documents (RAG), a tool call. Which gives the ladder every team ends up climbing:

1. **Prompt** it: a clear developer message with examples. Minutes, free.
2. **RAG**: put your documents in front of it. Hours, cheap.
3. **Tools**: let it fetch live data and take actions. Days.
4. **Fine-tune**: only when 1 to 3 are in place and the remaining problem is *how* it answers, at a volume where prompt tokens cost real money or latency matters.

One more reason fine-tuning sits last: a fine-tune is a snapshot of one base model, and base models are replaced every few months. Each new generation usually beats your old fine-tune out of the box, so the work has a short shelf life.

## The news: OpenAI is closing self-serve fine-tuning

As of September 2026, OpenAI's own deprecation page says it plainly. New organisations have not been able to create fine-tuning jobs since 7 May 2026; organisations without recent fine-tuned inference lost access on 2 July 2026; **on 6 January 2027 every remaining customer loses the ability to create new fine-tuning jobs**. Existing fine-tuned models keep serving until their base model retires. No GPT-5.x or GPT-6 model was ever fine-tunable; the last ones were the 2025 snapshots of gpt-4.1 (SFT and DPO) and o4-mini (reinforcement fine-tuning).

OpenAI's stated reason is the ladder above: newer base models follow instructions and formats well enough that prompting, RAG and tools cover the use cases fine-tuning used to serve, cheaper and faster. Other providers still offer it (Google, Together, Fireworks, and anyone with a GPU), so the technique is not gone. But the biggest API vendor deciding it is not worth running tells you where it sits on the list.

## How it was done on OpenAI (for the record)

Three steps, shown here as code you would have run, not as a live cell. The shape is the same on every hosted provider: a JSONL file of examples, a job, a new model id.

```python
# 1. Examples: one conversation per line, in the exact style you want (200 to 1,000 of these).
# tone.jsonl
{"messages": [{"role": "developer", "content": "You are TechStart support."},
              {"role": "user", "content": "My VPN drops every 10 minutes."},
              {"role": "assistant", "content": "Sorry about the VPN drops. Two quick checks: 1) ... 2) ... Reply here if neither helps. — Sam, TechStart IT"}]}
{"messages": [...]}

# 2. Upload the file and start the job.
training_file = client.files.create(file=open("tone.jsonl", "rb"), purpose="fine-tune")
job = client.fine_tuning.jobs.create(training_file=training_file.id, model="gpt-4.1-nano-2025-04-14")
# ... wait for job.status == "succeeded" (minutes to hours) ...

# 3. Use the new model id like any other model.
client.responses.create(model=job.fine_tuned_model, input="My VPN drops every 10 minutes.")
```

**SFT** (supervised fine-tuning) is the version above: show the right answer. **DPO** (direct preference optimisation) shows *pairs*, a better and a worse answer, and nudges the model toward the better one; same file format with `preferred_output` and `non_preferred_output`. Reinforcement fine-tuning goes further with a grader that scores answers; it was priced by the hour and only ever offered on o4-mini.

## What it costs: three worked examples

One training run: 1,000 examples of about 500 tokens each, 3 passes over the data (epochs). That is 1.5 million training tokens, a typical small job. Edit the numbers at the top and re-run.

In [4]:
examples, tokens_per_example, epochs = 1000, 500, 3
training_tokens = examples * tokens_per_example * epochs

# Published prices, September 2026 (per 1M training tokens)
prices = {
    "OpenAI gpt-4.1-nano (SFT, last available)": 1.50,
    "OpenAI gpt-4.1       (SFT, last available)": 25.00,
    "Together AI, 30B-class open model, LoRA SFT": 1.05,     # minimum charge $4 per job
}

print(f"Training tokens: {training_tokens/1e6:.1f}M\n")
for name, price in prices.items():
    cost = training_tokens / 1e6 * price
    note = "  (billed at the $4 minimum per job)" if "Together" in name and cost < 4 else ""
    print(f"{name:<46} ${cost:>7.2f}{note}")

Training tokens: 1.5M

OpenAI gpt-4.1-nano (SFT, last available)      $   2.25
OpenAI gpt-4.1       (SFT, last available)     $  37.50
Together AI, 30B-class open model, LoRA SFT    $   1.58  (billed at the $4 minimum per job)


### 🔍 What just happened?

Training is cheap: a few dollars for a small model, under $40 even for the largest one OpenAI ever offered. The real bill was *inference*: on OpenAI, a fine-tuned gpt-4.1-nano cost about twice the base model per token (input $0.20 versus $0.10 per million, output $0.80 versus $0.40), so a fine-tune that saves you a 300-token developer message on every request can still end up costing more than the prompt did. Do the arithmetic for your volume before you start.

## Doing it yourself on a 30B open model

The other route: rent a GPU and fine-tune an open model such as Qwen's 30B-class release (the naming moves fast: Qwen3-30B-A3B in 2025, Qwen 3.5 and 3.6 with 35B-A3B and a dense 27B in 2026). The question is always memory. The estimate below uses the usual rules of thumb for bytes per parameter; the measured number at the bottom comes from the Unsloth documentation.

In [5]:
params = 30e9

bytes_per_param = {
    "full fine-tuning, bf16 (weights + gradients + optimizer)": 16,
    "LoRA, bf16 weights frozen + small adapter":                 2.2,
    "QLoRA, 4-bit weights frozen + small adapter":               0.7,
}

print(f"Model: {params/1e9:.0f}B parameters\n")
for method, b in bytes_per_param.items():
    gb = params * b / 1e9
    fits = "multi-GPU server" if gb > 80 else ("A100/H100 80 GB" if gb > 32 else "one RTX 4090 / 5090")
    print(f"{method:<58} ~{gb:>5.0f} GB   -> {fits}")

print("\nMeasured (Unsloth docs, Qwen3-30B-A3B, QLoRA): about 17.5 GB of VRAM.")

Model: 30B parameters

full fine-tuning, bf16 (weights + gradients + optimizer)   ~  480 GB   -> multi-GPU server
LoRA, bf16 weights frozen + small adapter                  ~   66 GB   -> A100/H100 80 GB
QLoRA, 4-bit weights frozen + small adapter                ~   21 GB   -> one RTX 4090 / 5090

Measured (Unsloth docs, Qwen3-30B-A3B, QLoRA): about 17.5 GB of VRAM.


### 🔍 What just happened?

Full fine-tuning of a 30B model needs a rack of GPUs. LoRA brings it to one data-centre card; QLoRA brings it to a gaming GPU under your desk. That is why almost every open-model fine-tune you will see is a LoRA or QLoRA.

Renting instead of buying, at September 2026 prices: an A100 80 GB is roughly $1.40 to $2.80 per hour and an H100 about $2 to $4 (RunPod, Lambda). A QLoRA run over our 1.5M tokens takes on the order of an hour on either, so **one training run is a few dollars**, comparable to the hosted prices above. The expensive part is again what comes after: serving the model yourself means paying for the GPU whether anyone is asking questions or not.

Two practical notes. **OpenRouter**, the marketplace you use in the next course, is inference-only: it routes to models, it does not train them. Hosted fine-tuning for open models lives at Together AI and Fireworks (per training token) or in your own cloud account.

## When it is still worth it

Three situations where teams fine-tune in 2026 and are right to:

- **A fixed format at huge volume**: millions of classifications or extractions a day, where dropping a long developer message and few-shot examples from every request pays for the training many times over.
- **Latency**: a small fine-tuned model answering in 200 ms where a large prompted model takes 2 s.
- **Distillation**: a big model writes the perfect answers once, a small open model is fine-tuned to imitate them, and the small model runs on your own hardware for privacy or cost.

If your case is not one of these, climb the ladder first. Most helpdesk bots never reach rung four.

### 🎯 Your turn

Change `examples` to 5,000 and `epochs` to 4 in the cost cell and see whether the picture changes. Then increase the LoRA rank in the numpy demo (make `A` 4×2 and `B` 2×4) and check that it still learns, now with 16 trained numbers.

Sources: [OpenAI deprecations](https://developers.openai.com/api/docs/deprecations), [OpenAI pricing](https://developers.openai.com/api/docs/pricing), [LoRA paper](https://arxiv.org/abs/2106.09685), [QLoRA paper](https://arxiv.org/abs/2305.14314), [Unsloth Qwen3 guide](https://unsloth.ai/docs/models/tutorials/qwen3-how-to-run-and-fine-tune), [Together AI pricing](https://www.together.ai/pricing).